In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
import linear # type: ignore
import mse # type: ignore
from approx import approx # type: ignore
from common import repeat, test_model_grad, test_model_sgd, T # type: ignore

In [ ]:

class Per_Lin_MSE_Autograd(nn.Module):
    """
    A single-layer perceptron using `nn.Linear` for the affine transform, 
    and `nn.MSELoss` for training. 
    
    This is the highest-level, shortest, and fastest-to-implement version,
    relying fully on PyTorch to handle `forward` and `backward` passes.
    """
    

    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear =  nn.Linear(in_features, out_features)

    def forward(self, x):
        return self.model(x)
    
    def model(self, x):
        return self.linear(x)

    def loss(self, p, y):
        return nn.MSELoss(reduction='mean')(p, y)

    def predict(self, x):
        with tr.no_grad():
            return self.model(x)

In [ ]:

class Per_Lin_MSE_Backward(nn.Module):
    """
    A single-layer perceptron implemented using a fully custom autograd.Function 
    that defines both the forward pass and the analytical gradients for the `Linear`, 
    and `MSE` operations.
    
    The `forward` and `backward` passes for each operation are written manually, 
    but PyTorch's autograd still orchestrates the overall gradient flow 
    by traversing the computation graph from the final loss.

    This is the mid-level variant: lower-level than the pure-PyTorch version, 
    yet still leveraging autograd to call the custom backward functions.
    """

    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear =  linear.Linear(in_features, out_features)

    def forward(self, x):
        return self.model(x)
    
    def model(self, x):
        return self.linear(x)
    
    def loss(self, p, y):
        L = mse.MeanSquaredError()(p, y)
        return L

    def predict(self, x):
        with tr.no_grad():
            return self.model(x)    

$$ L = \frac{1}{N} \sum_{i=1}^{N}{(z_i - y_i) ^ 2} $$

$$ \frac{\partial L}{\partial z} = 2\frac{1}{N} (z - y) $$

$$ \diamond \diamond \diamond $$

$$ \text{Let z } = xW+b $$

$$ \frac{\partial L}{\partial W} = x^T \cdot \frac{\partial L}{\partial z} $$

$$ \frac{\partial L}{\partial b} = \sum_{i=1} \frac{\partial L}{\partial z_i} $$

In [ ]:

class Per_Lin_MSE_Gradient_Function(autograd.Function):
    """
    A single-layer perceptron implemented using a fully custom autograd.Function 
    that defines both the forward pass and the analytical gradients for the `Linear`, 
    and `MSE` operations.
    
    PyTorch's autograd no longer computes derivatives for intermediate steps — instead, 
    the entire gradient flow is produced by the manually written backward method.
    
    This is the lowest-level variant, exposing the exact mechanics of gradient computation 
    and giving full control over how the model participates in PyTorch's autograd system.
    """

    @staticmethod
    def _model(x, W, b):
        return linear.linear(x, W, b)
    
    @staticmethod
    def _loss(p, y):
        return mse.MeanSquaredError()(p, y)

    @staticmethod
    def predict(x, W, b):
        with tr.no_grad():
            return Per_Lin_MSE_Gradient_Function._model(x, W, b)

    @staticmethod
    def forward(ctx, x, W, b, y):
        z = Per_Lin_MSE_Gradient_Function._model(x, W, b)
        L = Per_Lin_MSE_Gradient_Function._loss(z, y)

        ctx.save_for_backward(x, W, y, z)

        return L
    
    @staticmethod
    def backward(ctx, dF_dL):
        x = ctx.saved_tensors[0]
        W = ctx.saved_tensors[1]
        y = ctx.saved_tensors[2]
        z = ctx.saved_tensors[3]

        S = x.shape[0]  # Samples
        FI = x.shape[1] # Features In
        FO = W.shape[1] # Features Out
        N = S * FO
        
        assert x.shape == (S, FI)
        assert W.shape == (FI, FO)
        assert y.shape == (S, FO)
        assert z.shape == (S, FO)

        dL_dz = 2 * (z - y)
        dL_dz /= N
        dL_dz *= dF_dL
        assert dL_dz.shape == (S, FO)
        
        # (FI, FO) = (S, FI).T @ (S, FO)
        dL_dW = x.t() @ dL_dz
        assert dL_dW.shape == (FI, FO)

        # `dL_dz` is already averaged over samples, 
        # so summing gives the correct gradient
        dL_db = dL_dz.sum(dim=0)
        assert dL_db.shape == (FO,)

        return (None, dL_dW, dL_db, None)


class Per_Lin_MSE_Gradient(nn.Module):
    """
    A single-layer perceptron implemented using a fully custom autograd.Function.
    """

    # In general the perceptron model is separable from the loss function, 
    # but since the total gradient is calculated manually, the loss function 
    # is an integrated part of it.
    #
    # This is helper for testing to indicate that the `forward` method expects `x` and `y`.

    CUSTOM_GRAD=True

    def __init__(self, in_features, out_features):
        super().__init__()

        self.weight = nn.Parameter(tr.randn(in_features, out_features, dtype=tr.float32))
        self.bias = nn.Parameter(tr.randn(out_features, dtype=tr.float32))

    def model(self, x):
        with tr.no_grad():
            return Per_Lin_MSE_Gradient_Function._model(x, self.weight, self.bias)
    
    def loss(self, p, y):
        with tr.no_grad():
            return Per_Lin_MSE_Gradient_Function._loss(p, y)

    def predict(self, x):
        with tr.no_grad():
            return Per_Lin_MSE_Gradient_Function.predict(x, self.weight, self.bias)
        
    def forward(self, x, y):
        assert x.shape[1] == self.weight.shape[0]
        assert y.shape[1] == self.weight.shape[1]

        return Per_Lin_MSE_Gradient_Function.apply(x, self.weight, self.bias, y)

In [ ]:
def test_linear_fn(model, a, b, samples=100, epochs=500, lr=0.1):
    x = tr.randn(samples, a.numel(), dtype=tr.float32)
    y = (x @ a).unsqueeze(1) + b
    return test_model_sgd(model, x, y, epochs, lr=lr)

    
if __name__ == "__main__":
    assert repeat(lambda: test_linear_fn(Per_Lin_MSE_Autograd(1, 1), T([1]).t(), T(1), samples=10, epochs=500)) >= 0.98
    assert repeat(lambda: test_linear_fn(Per_Lin_MSE_Backward(1, 1), T([1]).t(), T(1), samples=10, epochs=500)) >= 0.98
    assert repeat(lambda: test_linear_fn(Per_Lin_MSE_Gradient(1, 1), T([1]).t(), T(1), samples=10, epochs=500)) >= 0.98

    assert repeat(lambda: test_linear_fn(Per_Lin_MSE_Autograd(2, 1), T([1, 2]).t(), T(1), samples=10, epochs=500)) >= 0.98
    assert repeat(lambda: test_linear_fn(Per_Lin_MSE_Backward(2, 1), T([1, 2]).t(), T(1), samples=10, epochs=500)) >= 0.98
    assert repeat(lambda: test_linear_fn(Per_Lin_MSE_Gradient(2, 1), T([1, 2]).t(), T(1), samples=10, epochs=500)) >= 0.98

    assert repeat(lambda: test_linear_fn(Per_Lin_MSE_Autograd(3, 1), T([1, 2, 3]).t(), T(1), samples=10, epochs=500)) >= 0.98
    assert repeat(lambda: test_linear_fn(Per_Lin_MSE_Backward(3, 1), T([1, 2, 3]).t(), T(1), samples=10, epochs=500)) >= 0.98
    assert repeat(lambda: test_linear_fn(Per_Lin_MSE_Gradient(3, 1), T([1, 2, 3]).t(), T(1), samples=10, epochs=500)) >= 0.98
